# Combining S1 & S2 Flood layers

In [1]:
!pip install rasterio matplotlib geopandas numpy shapely


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 57.6 MB/s eta 0:00:00


In [ ]:
# Import Libraries
import logging
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import shapes
import geopandas as gpd
from pathlib import Path

## User Defined Config

In [ ]:
# Configuration
s1_tiff = Path('/content/flood_layerS1.tif')  # S1 flood layer
s2_tiff = Path('/content/flood_layerS2.tif')  # S2 flood layer
output_folder = Path('/content/combined_outputs')
output_folder.mkdir(parents=True, exist_ok=True)
s2_aligned_tiff = output_folder / 's2_aligned.tif'  # Aligned S2 TIFF
combined_tiff = output_folder / 's1_s2_final_flood.tif'
combined_shapefile = output_folder / 's1_s2_final_flood.shp'
target_crs = 'EPSG:25832'

## Saving vector and raster

In [24]:
from shapely.geometry import shape

# Set up logging
logging.basicConfig(filename='s1_s2_combine.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Verify inputs
if not s1_tiff.exists():
    logging.error("S1 TIFF missing.")
    raise FileNotFoundError("S1 TIFF missing.")
if not s2_tiff.exists():
    logging.error("S2 TIFF missing.")
    raise FileNotFoundError("S2 TIFF missing.")

# Align and combine flood layers
try:
    # Read S1 (reference grid)
    with rasterio.open(s1_tiff) as s1:
        s1_data = s1.read(1)
        s1_crs = s1.crs
        s1_transform = s1.transform
        s1_shape = s1_data.shape
        s1_nodata = s1.nodata if s1.nodata is not None else 0
        s1_valid = s1_data != s1_nodata
        s1_flood = s1_data > 0
        logging.info(f"S1: shape={s1_shape}, crs={s1_crs}, transform={s1_transform}, "
                     f"nodata={s1_nodata}, valid_pixels={np.sum(s1_valid)}, "
                     f"flood_pixels={np.sum(s1_flood)}")

    # Read S2 and reproject to S1's grid
    with rasterio.open(s2_tiff) as s2:
        s2_data = s2.read(1)
        s2_crs = s2.crs
        s2_transform = s2.transform
        s2_nodata = s2.nodata if s2.nodata is not None else 0
        s2_valid = s2_data != s2_nodata
        s2_flood = s2_data == 1
        logging.info(f"S2 (original): shape={s2_data.shape}, crs={s2_crs}, "
                     f"transform={s2_transform}, nodata={s2_nodata}, "
                     f"valid_pixels={np.sum(s2_valid)}, flood_pixels={np.sum(s2_flood)}")

        # Verify CRS
        if s1_crs != target_crs or s2_crs != target_crs:
            logging.error(f"CRS mismatch: S1={s1_crs}, S2={s2_crs}, Target={target_crs}")
            raise ValueError("CRS mismatch.")

        # Reproject S2 to S1's grid
        s2_aligned = np.zeros(s1_shape, dtype=np.uint8)
        reproject(
            source=s2_data,
            destination=s2_aligned,
            src_transform=s2_transform,
            src_crs=s2_crs,
            dst_transform=s1_transform,
            dst_crs=s1_crs,
            resampling=Resampling.nearest,  # Binary data
            src_nodata=s2_nodata,
            dst_nodata=s2_nodata
        )
        s2_aligned_valid = s2_aligned != s2_nodata
        s2_aligned_flood = s2_aligned == 1
        logging.info(f"S2 (aligned): shape={s2_aligned.shape}, "
                     f"valid_pixels={np.sum(s2_aligned_valid)}, "
                     f"flood_pixels={np.sum(s2_aligned_flood)}")

        # Save aligned S2 TIFF for debugging
        out_meta = s1.meta.copy()
        out_meta.update(dtype='uint8', nodata=s2_nodata)
        with rasterio.open(s2_aligned_tiff, 'w', **out_meta) as dst:
            dst.write(s2_aligned, 1)
        logging.info(f"Aligned S2 TIFF saved to {s2_aligned_tiff}")

    # Combine flood pixels (union)
    combined = np.zeros(s1_shape, dtype=np.uint8)
    combined[s1_valid & s1_flood] = 255
    combined[s2_aligned_valid & s2_aligned_flood] = 255
    logging.info(f"Combined: flood_pixels={np.sum(combined == 255)}")

    # Save combined TIFF
    out_meta.update(dtype='uint8', nodata=0)
    with rasterio.open(combined_tiff, 'w', **out_meta) as dst:
        dst.write(combined, 1)
    logging.info(f"Combined TIFF saved to {combined_tiff}")

    # Save shapefile
    results = shapes(combined, mask=combined == 255, transform=s1_transform)
    geometries = [shape(geom) for geom, _ in results]
    if geometries:
        gdf = gpd.GeoDataFrame({'geometry': geometries}, crs=target_crs)
        gdf.to_file(combined_shapefile)
        logging.info(f"Shapefile saved to {combined_shapefile} with CRS: {target_crs}")
    else:
        logging.warning("No flood pixels found; shapefile not created.")

except Exception as e:
    logging.error(f"Error combining flood layers: {str(e)}")
    raise